In [2]:
from cs336_basics import model as mt_model
from cs336_basics import BPETokenizer
import torch
from pathlib import Path
import time
import logging
import os

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

current_dir = Path(os.getcwd())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
tiny_vocab_path = str((current_dir / "../vocab_TinyStories.json").resolve())
tiny_merges_path = str((current_dir / "../merges_TinyStories.json").resolve())
tiny_tokenizer = BPETokenizer.from_pretrained(
    vocab_path=tiny_vocab_path,
    merges_path=tiny_merges_path,
    )
print(f"Tokenizer loaded from {tiny_vocab_path} and {tiny_merges_path}")
print(f"Tokenizer vocabs from 0 to 100: {[tiny_tokenizer.vocab[i] for i in range(100)]}")
print(f"Tokenizer <|endoftext|> id: {tiny_tokenizer.token2id[b"<|endoftext|>"]}")

2025-08-11 03:05:02 INFO cs336_basics.bpe_tokenizer: Vocab initialized: {0: b'<|endoftext|>', 1: b'\x00', 2: b'\x01', 3: b'\x02', 4: b'\x03', 5: b'\x04', 6: b'\x05', 7: b'\x06', 8: b'\x07', 9: b'\x08', 10: b'\t', 11: b'\n', 12: b'\x0b', 13: b'\x0c', 14: b'\r', 15: b'\x0e', 16: b'\x0f', 17: b'\x10', 18: b'\x11', 19: b'\x12', 20: b'\x13', 21: b'\x14', 22: b'\x15', 23: b'\x16', 24: b'\x17', 25: b'\x18', 26: b'\x19', 27: b'\x1a', 28: b'\x1b', 29: b'\x1c', 30: b'\x1d', 31: b'\x1e', 32: b'\x1f', 33: b' ', 34: b'!', 35: b'"', 36: b'#', 37: b'$', 38: b'%', 39: b'&', 40: b"'", 41: b'(', 42: b')', 43: b'*', 44: b'+', 45: b',', 46: b'-', 47: b'.', 48: b'/', 49: b'0', 50: b'1', 51: b'2', 52: b'3', 53: b'4', 54: b'5', 55: b'6', 56: b'7', 57: b'8', 58: b'9', 59: b':', 60: b';', 61: b'<', 62: b'=', 63: b'>', 64: b'?', 65: b'@', 66: b'A', 67: b'B', 68: b'C', 69: b'D', 70: b'E', 71: b'F', 72: b'G', 73: b'H', 74: b'I', 75: b'J', 76: b'K', 77: b'L', 78: b'M', 79: b'N', 80: b'O', 81: b'P', 82: b'Q', 83: b

Tokenizer loaded from /data/satori_hdd1/mutyuu/workspace/CS336/assignment1/experiments/vocab_TinyStories.json and /data/satori_hdd1/mutyuu/workspace/CS336/assignment1/experiments/merges_TinyStories.json
Tokenizer vocabs from 0 to 100: [b'<|endoftext|>', b'\x00', b'\x01', b'\x02', b'\x03', b'\x04', b'\x05', b'\x06', b'\x07', b'\x08', b'\t', b'\n', b'\x0b', b'\x0c', b'\r', b'\x0e', b'\x0f', b'\x10', b'\x11', b'\x12', b'\x13', b'\x14', b'\x15', b'\x16', b'\x17', b'\x18', b'\x19', b'\x1a', b'\x1b', b'\x1c', b'\x1d', b'\x1e', b'\x1f', b' ', b'!', b'"', b'#', b'$', b'%', b'&', b"'", b'(', b')', b'*', b'+', b',', b'-', b'.', b'/', b'0', b'1', b'2', b'3', b'4', b'5', b'6', b'7', b'8', b'9', b':', b';', b'<', b'=', b'>', b'?', b'@', b'A', b'B', b'C', b'D', b'E', b'F', b'G', b'H', b'I', b'J', b'K', b'L', b'M', b'N', b'O', b'P', b'Q', b'R', b'S', b'T', b'U', b'V', b'W', b'X', b'Y', b'Z', b'[', b'\\', b']', b'^', b'_', b'`', b'a', b'b']
Tokenizer <|endoftext|> id: 0


In [8]:
model_config = {
    "d_model": 512,
    "num_heads": 16,
    "d_ff": 1344,
    "num_layers": 4,
    "vocab_size": 10000,
    "max_seq_len": 256,
    "theta": 10000.0,
}
tiny_model = mt_model.Transformer(
    model_config["d_model"],
    model_config["num_heads"],
    model_config["d_ff"],
    model_config["num_layers"],
    model_config["vocab_size"],
    model_config["max_seq_len"],
    model_config["theta"],
)
tiny_model.to(device)
mt_model.load_checkpoint(
    src="checkpoints1/checkpoint_iter_5000.pth",
    model=tiny_model,
    optimizer=None,
)

5000

In [9]:
prompt = "I hate Mondays."
prompt_ids = tiny_tokenizer.encode(prompt)
prompt_tensor = torch.tensor(prompt_ids, dtype=torch.long).unsqueeze(0).to(device)
generated_ids = tiny_model.generate(
    input_ids=prompt_tensor,
    max_length=256,
    temperature=1.0,
    top_p=0.9,
)
print(f"Prompt: {prompt}")
print(f"Prompt IDs: {prompt_ids}")
print(f"Generated IDs: {generated_ids.tolist()}")
generated_text = tiny_tokenizer.decode(generated_ids[0].tolist())
print(f"Generated text: {generated_text}")

Prompt: I hate Mondays.
Prompt IDs: [74, 4261, 379, 2856, 5813, 47]
Generated IDs: [[74, 4261, 379, 2856, 5813, 47, 974, 754, 440, 431, 802, 2400, 391, 876, 47, 796, 483, 854, 1246, 266, 570, 381, 985, 397, 11, 1675, 722, 384, 1142, 283, 927, 2880, 47, 316, 427, 354, 259, 737, 2649, 267, 382, 263, 887, 893, 1358, 47, 316, 3000, 266, 263, 985, 381, 309, 1416, 322, 259, 850, 985, 658, 283, 1434, 313, 259, 277, 434, 335, 653, 118, 47, 670, 3733, 263, 327, 286, 529, 733, 823, 839, 286, 652, 342, 690, 47, 11, 2337, 263, 887, 893, 826, 579, 263, 985, 267, 548, 266, 1254, 1302, 47, 285, 985, 7253, 880, 267, 432, 263, 278, 1649, 273, 3505, 267, 265, 1309, 1496, 267, 1975, 313, 1234, 110, 2856, 267, 4499, 47, 341, 283, 391, 376, 267, 924, 342, 722, 47, 316, 672, 381, 726, 327, 286, 529, 322, 259, 823, 2514, 658, 283, 1061, 1014, 1048, 267, 965, 47, 11, 0]]
Generated text: I hate Mondays. Pogry is just working so hard. We are never here to find that boat."
His dad's work was still difficult. He 